# EdgeVerify — On-Device / Low-Power Experiments

Two parts: **(A)** memory-bounded training (firebreak) — run here on a **GPU runtime**; **(B)** export the encoder to ONNX and benchmark it **on your Android phone** via Termux.

See `docs/on_device_plan.md` in the repo for the pre-registered criteria.


## 0. Install


In [ ]:
!pip install -q torch onnx

## 1. Files


In [ ]:
%%writefile model.py
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F

def compute_stable_local_loss(projected_student, teacher_target, layer_output,
                              variance_threshold=1.0, alpha=1.0, beta=0.01,
                              shared_cov=False, var_on="context", cov_on=None,
                              return_components=False):
    """
    Computes regularized local distillation loss using VicReg-style constraints
    to prevent dimensional collapse.

    The default coefficients (alpha=1.0, beta=0.01) reproduce the original
    formulation exactly. They are exposed as arguments so that the
    unregularized baseline (alpha=beta=0) required for the collapse ablation
    can share this single code path. Set return_components=True to obtain the
    per-term breakdown for logging.

    Placement of the two regularizers is configurable, to support the
    regularizer-placement study. ``var_on`` and ``cov_on`` each select the
    tensor a penalty acts on: ``"context"`` = the context embedding
    ``layer_output`` (s_t), ``"pred"`` = the predictor output
    ``projected_student`` (s_pred).

    Backward compatibility: ``var_on`` defaults to ``"context"``. If ``cov_on``
    is left as ``None`` it is resolved from ``shared_cov`` -- ``"context"`` when
    ``shared_cov=True`` (the shared-embedding variant), otherwise ``"pred"``
    (the original wiring). An explicit ``cov_on`` overrides ``shared_cov``.
    """
    tensors = {"context": layer_output, "pred": projected_student}
    if cov_on is None:
        cov_on = "context" if shared_cov else "pred"
    var_tensor = tensors[var_on]
    cov_tensor = tensors[cov_on]

    # 1. Base Distillation Loss (MSE against target)
    distill_loss = F.mse_loss(projected_student, teacher_target)

    # 2. Variance Constraint (Hinge loss on batch standard deviation)
    std_student = torch.sqrt(var_tensor.var(dim=0) + 1e-4)
    variance_loss = torch.mean(F.relu(variance_threshold - std_student))

    # 3. Covariance Regularization (Feature decorrelation)
    centered_student = cov_tensor - cov_tensor.mean(dim=0)
    batch_size = cov_tensor.size(0)
    cov_matrix = (centered_student.T @ centered_student) / (batch_size - 1)
    diag_mask = torch.eye(cov_matrix.size(0), device=cov_tensor.device)
    covariance_loss = (cov_matrix * (1 - diag_mask)).pow(2).sum() / cov_matrix.size(0)

    total = distill_loss + alpha * variance_loss + beta * covariance_loss
    if return_components:
        return total, {
            "distill": distill_loss.item(),
            "var": variance_loss.item(),
            "cov": covariance_loss.item(),
            "total": total.item(),
        }
    return total

def _build_encoder(backbone, img_channels, latent_dim):
    """Build the context/target encoder. ``backbone='cnn'`` is the original
    lightweight 3-block CNN (default, so prior results are unchanged);
    ``backbone='resnet18'`` uses a randomly-initialized torchvision ResNet-18
    with its classifier head replaced by a linear projection to ``latent_dim``,
    for the larger-scale study."""
    if backbone == "cnn":
        return nn.Sequential(
            nn.Conv2d(img_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, latent_dim, kernel_size=3, stride=2, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )
    if backbone == "resnet18":
        import torchvision
        net = torchvision.models.resnet18(weights=None)  # random init, no download
        if img_channels != 3:
            net.conv1 = nn.Conv2d(img_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        net.fc = nn.Linear(512, latent_dim)  # 512 = ResNet-18 penultimate width
        return net
    raise ValueError(f"Unknown backbone: {backbone!r}")


class VisionJEPA(nn.Module):
    def __init__(self, img_channels=3, latent_dim=256, ema_decay=0.999, backbone="cnn"):
        super().__init__()
        self.latent_dim = latent_dim
        self.ema_decay = ema_decay
        self.backbone = backbone

        # Context Encoder (f_theta)
        self.context_encoder = _build_encoder(backbone, img_channels, latent_dim)

        # Target Encoder (f_theta_bar) - Updated via EMA
        self.target_encoder = copy.deepcopy(self.context_encoder)
        for p in self.target_encoder.parameters():
            p.requires_grad = False

        # Action/Bounding-Box Encoder
        self.action_encoder = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )

        # Latent Predictor Block (p_psi)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, 512),
            nn.ReLU(),
            nn.Linear(512, latent_dim)
        )

    @torch.no_grad()
    def update_target_encoder(self):
        for p_ctx, p_tgt in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            p_tgt.data.mul_(self.ema_decay).add_(p_ctx.data, alpha=1.0 - self.ema_decay)

    def forward(self, partial_images, full_images, spatial_actions):
        s_t = self.context_encoder(partial_images)
        with torch.no_grad():
            s_target = self.target_encoder(full_images)
        a_t = self.action_encoder(spatial_actions)

        combined_latent = torch.cat([s_t, a_t], dim=-1)
        s_predicted = self.predictor(combined_latent)

        return s_predicted, s_target, s_t


In [ ]:
%%writefile evaluate_memory.py
"""
Memory-bounded on-device training: end-to-end backprop vs. layer-wise local
("firebreak") training.

A depth-configurable encoder is trained two ways with a VICReg-style
variance/covariance objective:

  * end-to-end : one global objective on the final embedding; the whole
    network's activation graph is retained for backprop, so peak activation
    memory grows with depth.

  * layer-wise local ("firebreak") : each block has its own local objective and
    is optimized in isolation, with the block output detached before the next
    block. Only one block's activation graph is live at a time, so peak
    activation memory is (approximately) independent of depth. This is the
    on-device training scheme the thesis motivates.

For each depth we report peak training memory (real ``torch.cuda`` peak on GPU;
a portable analytical activation-memory estimate otherwise) and the effective
rank of the learned final embedding (to confirm the local scheme still learns
structured, non-collapsed features).

Run on a Colab GPU for the real peak-memory numbers; it also runs on CPU/laptop
(reporting the analytical estimate).
"""

import json, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEPTHS = [2, 4, 8, 16, 24]
C = 64          # channels per block
HW = 32         # spatial size kept constant across blocks
BATCH = 64
STEPS = 30
LATENT = 128


def make_blocks(depth):
    stem = nn.Sequential(nn.Conv2d(3, C, 3, stride=2, padding=1), nn.BatchNorm2d(C), nn.ReLU())
    blocks = nn.ModuleList([
        nn.Sequential(nn.Conv2d(C, C, 3, padding=1), nn.BatchNorm2d(C), nn.ReLU())
        for _ in range(depth)
    ])
    head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(C, LATENT))
    return stem, blocks, head


def vicreg(z, gamma=1.0):
    std = torch.sqrt(z.var(dim=0) + 1e-4)
    var_loss = torch.mean(F.relu(gamma - std))
    zc = z - z.mean(0, keepdim=True)
    cov = (zc.T @ zc) / (z.size(0) - 1)
    off = (cov * (1 - torch.eye(cov.size(0), device=z.device))).pow(2).sum() / cov.size(0)
    return var_loss + off


def local_heads(depth, device):
    return nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                        nn.Linear(C, LATENT)) for _ in range(depth)]).to(device)


def activation_bytes(depth, mode):
    """Portable estimate of retained activation memory (float32) for one step."""
    per_block = BATCH * C * HW * HW * 4
    return (depth * per_block) if mode == "e2e" else per_block  # e2e retains all; local one


def run(depth, mode, device):
    torch.manual_seed(0)
    stem, blocks, head = make_blocks(depth)
    stem, blocks, head = stem.to(device), blocks.to(device), head.to(device)
    params = list(stem.parameters()) + list(blocks.parameters()) + list(head.parameters())
    lheads = local_heads(depth, device) if mode == "local" else None
    if mode == "local":
        params += list(lheads.parameters())
    opt = torch.optim.AdamW(params, lr=1e-3)

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for _ in range(STEPS):
        x = torch.rand(BATCH, 3, 64, 64, device=device)
        opt.zero_grad()
        h = stem(x)
        if mode == "e2e":
            for b in blocks:
                h = b(h)
            z = head(h)
            loss = vicreg(z)
            loss.backward()
        else:  # layer-wise local firebreak
            for i, b in enumerate(blocks):
                h = b(h)
                z = lheads[i](h)
                vicreg(z).backward()      # confined to this block
                h = h.detach()            # firebreak: cut the graph
            z = head(h.detach())          # final embedding for evaluation only
        opt.step()

    # Final embedding effective rank (eval)
    with torch.no_grad():
        h = stem(torch.rand(256, 3, 64, 64, device=device))
        for b in blocks:
            h = b(h)
        z = head(h)
        zc = z - z.mean(0, keepdim=True)
        cov = (zc.T @ zc) / (z.size(0) - 1)
        eig = torch.linalg.eigvalsh(cov).clamp(min=0)
        rank = (eig.sum() ** 2 / (eig ** 2).sum()).item()

    if device.type == "cuda":
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        metric = "peak_gpu_mb"
    else:
        peak_mb = activation_bytes(depth, mode) / (1024 ** 2)
        metric = "est_activation_mb"
    return peak_mb, metric, rank


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device={device}\n", flush=True)
    results = {}
    for depth in DEPTHS:
        row = {}
        for mode in ("e2e", "local"):
            try:
                peak, metric, rank = run(depth, mode, device)
                row[mode] = {metric: round(peak, 1), "eff_rank": round(rank, 2)}
                print(f"depth={depth:2d} {mode:5s}  {metric}={peak:8.1f}  eff_rank={rank:.2f}", flush=True)
            except RuntimeError as e:  # e.g. CUDA OOM for deep e2e
                row[mode] = {"error": str(e)[:80]}
                print(f"depth={depth:2d} {mode:5s}  ERROR: {str(e)[:60]}", flush=True)
        results[depth] = row
    with open("memory_bounded.json", "w") as f:
        json.dump({"channels": C, "hw": HW, "batch": BATCH, "results": results}, f, indent=2)
    print("DONE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export_onnx.py
"""
Export the trained context encoder to ONNX for on-device (phone) benchmarking.

Produces ``encoder.onnx`` containing the context encoder (the part that runs at
inference time on the edge device). By default it exports a randomly-initialized
CNN encoder; pass a checkpoint to export trained weights.
"""

import argparse
import torch
from model import VisionJEPA


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--backbone", default="cnn", choices=["cnn", "resnet18"])
    ap.add_argument("--ckpt", default=None, help="optional state_dict for the full VisionJEPA")
    ap.add_argument("--out", default="encoder.onnx")
    args = ap.parse_args()

    model = VisionJEPA(backbone=args.backbone)
    if args.ckpt:
        model.load_state_dict(torch.load(args.ckpt, map_location="cpu"))
    encoder = model.context_encoder.eval()

    dummy = torch.rand(1, 3, 64, 64)
    torch.onnx.export(
        encoder, dummy, args.out,
        input_names=["image"], output_names=["embedding"],
        dynamic_axes={"image": {0: "batch"}, "embedding": {0: "batch"}},
        opset_version=17, dynamo=False,
    )
    n = sum(p.numel() for p in encoder.parameters())
    print(f"exported {args.out}  |  backbone={args.backbone}  params={n:,}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile bench_phone.py
"""
On-device (phone) inference benchmark for the exported encoder.

Runs ``encoder.onnx`` with ONNX Runtime and reports single-image inference
latency, throughput, model size, and peak process memory. Designed to run on an
Android phone via Termux (real ARM-CPU edge numbers), and works on a laptop too.

--- HOW TO RUN ON AN ANDROID PHONE (Termux) ---
  1. Install the Termux app (F-Droid build recommended).
  2. In Termux:
        pkg update && pkg install python
        pip install onnxruntime numpy
  3. Copy encoder.onnx and this script into Termux storage
     (e.g. `termux-setup-storage` then place them under ~/storage/shared, or scp).
  4. Run:
        python bench_phone.py --onnx encoder.onnx --runs 200
  5. Record the printed latency / memory. For a rough energy figure, note the
     battery percentage before and after a long run (e.g. --runs 20000).
"""

import argparse
import os
import time

import numpy as np


def peak_rss_mb():
    try:
        import resource
        # ru_maxrss is KB on Linux/Android, bytes on macOS
        kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return kb / 1024.0 if kb > 1e6 else kb / 1024.0
    except Exception:
        return float("nan")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--onnx", default="encoder.onnx")
    ap.add_argument("--runs", type=int, default=200)
    ap.add_argument("--warmup", type=int, default=20)
    ap.add_argument("--threads", type=int, default=0, help="0 = ORT default")
    args = ap.parse_args()

    import onnxruntime as ort
    so = ort.SessionOptions()
    if args.threads > 0:
        so.intra_op_num_threads = args.threads
    sess = ort.InferenceSession(args.onnx, sess_options=so, providers=["CPUExecutionProvider"])
    iname = sess.get_inputs()[0].name

    x = np.random.rand(1, 3, 64, 64).astype(np.float32)
    for _ in range(args.warmup):
        sess.run(None, {iname: x})

    lat = []
    for _ in range(args.runs):
        t0 = time.perf_counter()
        sess.run(None, {iname: x})
        lat.append((time.perf_counter() - t0) * 1000.0)
    lat = np.array(lat)

    size_mb = os.path.getsize(args.onnx) / (1024 ** 2)
    print(f"model            : {args.onnx} ({size_mb:.2f} MB on disk)")
    print(f"runs             : {args.runs} (warmup {args.warmup})")
    print(f"latency (ms)     : mean {lat.mean():.2f}  std {lat.std():.2f}  "
          f"p50 {np.percentile(lat,50):.2f}  p90 {np.percentile(lat,90):.2f}")
    print(f"throughput (img/s): {1000.0/lat.mean():.1f}")
    print(f"peak process RSS : {peak_rss_mb():.1f} MB")


if __name__ == "__main__":
    main()


## A. Memory-bounded training (firebreak) — use a GPU runtime

Reports peak **training** memory of end-to-end vs. layer-wise-local backprop as encoder depth grows. On GPU you get real `torch.cuda` peak memory. Expectation: `e2e` grows with depth (and may OOM at large depth); `local` stays roughly flat, while its embedding effective rank stays > 1 (not collapsed).


In [ ]:
!python evaluate_memory.py

### Plot the memory-vs-depth curve (after the run above)


In [ ]:
import json, matplotlib.pyplot as plt
r=json.load(open("memory_bounded.json"))["results"]
depths=sorted(int(d) for d in r)
key=lambda d,m:[k for k in r[str(d)][m] if k.endswith("_mb")][0]
e2e=[r[str(d)]["e2e"].get(key(d,"e2e")) if "e2e" in r[str(d)] and any(k.endswith("_mb") for k in r[str(d)]["e2e"]) else None for d in depths]
loc=[r[str(d)]["local"].get(key(d,"local")) if any(k.endswith("_mb") for k in r[str(d)]["local"]) else None for d in depths]
plt.figure(figsize=(6,4))
plt.plot(depths,e2e,"o-",label="end-to-end"); plt.plot(depths,loc,"s-",label="layer-wise local (firebreak)")
plt.xlabel("encoder depth (blocks)"); plt.ylabel("peak training memory (MB)")
plt.title("Firebreak keeps training memory flat as depth grows"); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## B. Export the encoder to ONNX (download for your phone)


In [ ]:
!python export_onnx.py --backbone cnn --out encoder.onnx
from google.colab import files
files.download('encoder.onnx')  # save it, then move to your phone

## B. Benchmark on your Android phone (Termux)

1. Install **Termux** (F-Droid build).
2. In Termux: `pkg install python` then `pip install onnxruntime numpy`.
3. Put `encoder.onnx` and `bench_phone.py` in Termux (e.g. `termux-setup-storage`, then copy from shared storage).
4. Run: `python bench_phone.py --onnx encoder.onnx --runs 200`
5. Record latency / memory. For rough energy, note battery % before/after `--runs 20000`.

(You can also run `bench_phone.py` here as a laptop/CPU proxy, but the *phone* numbers are the edge result.)


In [ ]:
!python bench_phone.py --onnx encoder.onnx --runs 100   # laptop/CPU proxy; real numbers come from the phone